In [11]:
pip install openpyxl

In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Fetch full Wikipedia pages for MEPs in their national languages and in English,
then save to CSV and XLSX, with robust fuzzy title resolution.

Reads the FIRST 10 MEPs from an Excel file (columns: full_name, country).

Outputs:
- mep_wikipedia_pages.csv  (UTF-8 with BOM for Excel)
- mep_wikipedia_pages.xlsx  (via openpyxl)
"""

from __future__ import annotations
import csv
import logging
import os
import time
from typing import Optional, Tuple, List, Dict
import unicodedata
import re
import difflib
from urllib.parse import quote

import pandas as pd
import requests
import wikipediaapi

# -------- Configuration --------

USER_AGENT = "MEP-Wiki-Scraper/1.5 (contact: you@example.com)"
OUTPUT_CSV = "mep_wikipedia_pages.csv"
OUTPUT_XLSX = "mep_wikipedia_pages.xlsx"

# Path to your Excel file
EXCEL_PATH = r"C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\Member Data\1729084703_MEP_list Oct 2024.xlsx"

REQUEST_PAUSE_SECONDS = 0.5
HEADERS = {"User-Agent": USER_AGENT}
SEARCH_LIMIT = 10

# -------- Country -> Wikipedia language --------

COUNTRY_TO_LANG = {
    "Finland": "fi",
    "Spain": "es",
    "Poland": "pl",
    "Greece": "el",
    "Malta": "mt",
    "France": "fr",
    "Germany": "de",
    "Italy": "it",
    "Portugal": "pt",
    "Netherlands": "nl",
    "Belgium": "fr",
    "Sweden": "sv",
    "Denmark": "da",
    "Czechia": "cs",
    "Czech Republic": "cs",
    "Slovakia": "sk",
    "Slovenia": "sl",
    "Austria": "de",
    "Ireland": "ga",
    "Romania": "ro",
    "Bulgaria": "bg",
    "Hungary": "hu",
    "Croatia": "hr",
    "Lithuania": "lt",
    "Latvia": "lv",
    "Estonia": "et",
    "Cyprus": "el",
    "Luxembourg": "lb",
}

# -------- Load MEPs from Excel --------

def load_meps_from_excel(path: str, n: int = 999999) -> List[dict]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Excel file not found at: {path}")
    df = pd.read_excel(path, dtype=str)
    for col in ["full_name", "country"]:
        if col not in df.columns:
            raise KeyError(f"Required column '{col}' not found in Excel. Present columns: {list(df.columns)}")

    df = df[["full_name", "country"]].head(n).copy()
    df["full_name"] = df["full_name"].fillna("").astype(str).str.strip()
    df["country"] = df["country"].fillna("").astype(str).str.strip()
    df = df[(df["full_name"] != "") & (df["country"] != "")]
    return [{"fullName": r.full_name, "country": r.country} for r in df.itertuples(index=False)]

# -------- Helpers --------

def strip_diacritics(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")

def simplify(s: str) -> str:
    s = strip_diacritics(s.lower())
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalise_name_case(name: str) -> str:
    parts = name.split()
    return " ".join(p.capitalize() if p.isupper() else p for p in parts)

def generate_name_variants(name: str) -> List[str]:
    variants: List[str] = []
    base = name.strip()

    def add(v: str):
        if v and v not in variants:
            variants.append(v)

    add(base)
    add(normalise_name_case(base))
    no_diac = strip_diacritics(base)
    add(no_diac)
    add(normalise_name_case(no_diac))
    add(" ".join(base.split()))

    tokens = base.split()
    if len(tokens) >= 2:
        first, last = tokens[0], tokens[-1]
        add(f"{first} {last}")
        add(f"{last} {first}")
        add(strip_diacritics(f"{first} {last}"))
        add(strip_diacritics(f"{last} {first}"))
        add(normalise_name_case(f"{first} {last}"))
        add(normalise_name_case(f"{last} {first}"))

    # Greek common variant: Georgios <-> Giorgos, Georgios <-> Yorgos
    replacements = [
        ("Georgios", "Giorgos"),
        ("Giorgos", "Georgios"),
        ("Georgios", "Yorgos"),
        ("Yorgos", "Georgios"),
    ]
    for a, b in replacements:
        if a in base:
            add(base.replace(a, b))
            add(normalise_name_case(base.replace(a, b)))

    return variants

def mw_query(lang: str, params: Dict) -> Dict:
    base = f"https://{lang}.wikipedia.org/w/api.php"
    r = requests.get(base, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def mw_exact_title_if_exists(lang: str, title: str) -> Optional[str]:
    data = mw_query(lang, {
        "action": "query",
        "format": "json",
        "prop": "info",
        "redirects": "1",
        "titles": title,
        "inprop": "url",
    })
    pages = data.get("query", {}).get("pages", {})
    for pid, page in pages.items():
        if str(pid).lstrip("-").isdigit() and int(pid) > 0 and "missing" not in page:
            return page.get("title")
    return None

def mw_search_candidates(lang: str, query: str, limit: int = SEARCH_LIMIT) -> List[str]:
    titles: List[str] = []

    data = mw_query(lang, {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
        "srlimit": str(limit),
        "utf8": "1",
        "srprop": "",
    })
    for hit in data.get("query", {}).get("search", []):
        t = hit.get("title")
        if t and t not in titles:
            titles.append(t)

    data2 = mw_query(lang, {
        "action": "query",
        "format": "json",
        "list": "prefixsearch",
        "pssearch": query,
        "pslimit": str(limit),
    })
    for hit in data2.get("query", {}).get("prefixsearch", []):
        t = hit.get("title")
        if t and t not in titles:
            titles.append(t)

    return titles

def score_title(candidate: str, target_name: str) -> float:
    a = simplify(candidate)
    b = simplify(target_name)
    base = difflib.SequenceMatcher(None, a, b).ratio()
    if a.startswith(b) or b.startswith(a):
        base += 0.15
    ta = set(a.split())
    tb = set(b.split())
    if ta and tb:
        overlap = len(ta & tb) / max(len(ta), 1)
        base += 0.2 * overlap
    return min(base, 1.0)

def resolve_title_fuzzy(lang: str, name: str) -> Optional[str]:
    # quick exact passes on variants
    for cand in generate_name_variants(name):
        exact = mw_exact_title_if_exists(lang, cand)
        if exact:
            return exact

    # candidate pool from searches
    pool: List[str] = []
    queries = []
    for cand in generate_name_variants(name):
        queries.extend([f'"{cand}"', f'intitle:"{cand}"', cand])

    for q in queries:
        for t in mw_search_candidates(lang, q, limit=SEARCH_LIMIT):
            if t not in pool:
                pool.append(t)
        if len(pool) >= SEARCH_LIMIT:
            break

    if not pool:
        return None

    best = max(pool, key=lambda t: score_title(t, name))
    if score_title(best, name) >= 0.62:
        return best
    return None

def fetch_html_and_wikitext(lang: str, title: str) -> Tuple[str, str, str, Dict[str, dict]]:
    # HTML + langlinks
    data = mw_query(lang, {
        "action": "parse",
        "page": title,
        "prop": "text|displaytitle|langlinks",
        "format": "json",
        "formatversion": "2",
        "redirects": "1",
        "lllimit": "500",
    })
    if "parse" not in data:
        return "", "", "", {}

    html = data["parse"].get("text") or ""
    display_title = data["parse"].get("title") or title
    url = f"https://{lang}.wikipedia.org/wiki/{quote(display_title.replace(' ', '_'))}"

    langlinks_raw = data["parse"].get("langlinks") or []
    langlinks: Dict[str, dict] = {}
    for ll in langlinks_raw:
        code = ll.get("lang")
        if not code:
            continue
        ll_title = ll.get("title") or ll.get("*") or ""
        ll_url = ll.get("url") or (f"https://{code}.wikipedia.org/wiki/{quote(ll_title.replace(' ', '_'))}" if ll_title else "")
        langlinks[code] = {"title": ll_title, "url": ll_url}

    # wikitext
    data_w = mw_query(lang, {
        "action": "parse",
        "page": title,
        "prop": "wikitext",
        "format": "json",
        "formatversion": "2",
        "redirects": "1",
    })
    wikitext_val = ""
    if "parse" in data_w:
        wt = data_w["parse"].get("wikitext")
        if isinstance(wt, str):
            wikitext_val = wt
        elif isinstance(wt, dict):
            wikitext_val = wt.get("*", "") or wt.get("content", "")
        else:
            wikitext_val = ""

    return html, wikitext_val, url, langlinks

def resolve_via_english_bridge(native_lang: str, name: str) -> Optional[str]:
    en_title = resolve_title_fuzzy("en", name)
    if not en_title:
        return None
    _, _, _, ll = fetch_html_and_wikitext("en", en_title)
    native = ll.get(native_lang)
    if native and native.get("title"):
        return native["title"]
    return None

# -------- Independent fetchers: always try both sides --------

def fetch_native_side(lang: Optional[str], name: str) -> Tuple[str, str, str, str, Dict[str, dict]]:
    """
    Returns: page_title_native, page_url_native, html_native, wikitext_native, langlinks_native
    Empty strings if not found or no language provided.
    """
    if not lang:
        return "", "", "", "", {}
    title_native = resolve_title_fuzzy(lang, name)
    if not title_native:
        # try to find native via english langlinks as a bridge
        bridged = resolve_via_english_bridge(lang, name)
        if bridged:
            title_native = bridged
    if not title_native:
        return "", "", "", "", {}
    html_native, wiki_native, url_native, langlinks = fetch_html_and_wikitext(lang, title_native)
    return title_native or "", url_native or "", html_native or "", wiki_native or "", langlinks

def fetch_english_side(name: str, langlinks_from_native: Optional[Dict[str, dict]] = None) -> Tuple[str, str, str, str]:
    """
    Returns: page_title_en, page_url_en, html_en, wikitext_en
    Uses native langlinks if available, else resolves independently.
    """
    page_title_en = ""
    page_url_en = ""
    html_en = ""
    wiki_en = ""

    if langlinks_from_native and "en" in langlinks_from_native and langlinks_from_native["en"].get("title"):
        page_title_en = langlinks_from_native["en"]["title"] or ""
        page_url_en = langlinks_from_native["en"].get("url") or ""

    if not page_title_en:
        maybe_en = resolve_title_fuzzy("en", name)
        if maybe_en:
            page_title_en = maybe_en

    if page_title_en:
        html_en, wiki_en, url_en, _ = fetch_html_and_wikitext("en", page_title_en)
        if url_en:
            page_url_en = url_en

    return page_title_en or "", page_url_en or "", html_en or "", wiki_en or ""

# -------- Main --------

def main() -> None:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

    meps_from_file = load_meps_from_excel(EXCEL_PATH, n=99999)
    if not meps_from_file:
        raise RuntimeError("No MEP rows loaded from Excel. Check columns 'full_name' and 'country'.")

    rows = []

    for mep in meps_from_file:
        name = mep["fullName"].strip()
        country = mep["country"].strip()
        lang = COUNTRY_TO_LANG.get(country)

        logging.info("Processing %s [%s -> %s + en]", name, country, lang or "N/A")

        # Always try native and English independently
        page_title_native, page_url_native, html_native, wiki_native, langlinks_native = fetch_native_side(lang, name)
        page_title_en, page_url_en, html_en, wiki_en = fetch_english_side(name, langlinks_from_native=langlinks_native if langlinks_native else None)

        if not html_native and not wiki_native:
            logging.info("Native page not found or empty for %s", name)
        if not html_en and not wiki_en:
            logging.info("English page not found or empty for %s", name)

        rows.append(
            {
                "name": name,
                "wikipedia_language": lang or "",
                "page_title": page_title_native,
                "page_url": page_url_native,
                "wikipedia_content_html": html_native,
                "wikipedia_content_wikitext": wiki_native,
                "page_title_en": page_title_en,
                "page_url_en": page_url_en,
                "wikipedia_content_html_en": html_en,
                "wikipedia_content_wikitext_en": wiki_en,
            }
        )

        time.sleep(REQUEST_PAUSE_SECONDS)

    df = pd.DataFrame(
        rows,
        columns=[
            "name",
            "wikipedia_language",
            "page_title",
            "page_url",
            "wikipedia_content_html",
            "wikipedia_content_wikitext",
            "page_title_en",
            "page_url_en",
            "wikipedia_content_html_en",
            "wikipedia_content_wikitext_en",
        ],
    )

    df.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8-sig")
    df.to_excel(OUTPUT_XLSX, index=False)

    print(f"Saved {len(df)} rows to {OUTPUT_CSV} and {OUTPUT_XLSX}")

if __name__ == "__main__":
    main()


INFO: Processing Mika AALTOLA [Finland -> fi + en]
INFO: Processing Maravillas ABADÍA JOVER [Spain -> es + en]
INFO: Processing Magdalena ADAMOWICZ [Poland -> pl + en]
INFO: Processing Georgios AFTIAS [Greece -> el + en]
INFO: Processing Oihane AGIRREGOITIA MARTÍNEZ [Spain -> es + en]
INFO: Processing Peter AGIUS [Malta -> mt + en]
INFO: Native page not found or empty for Peter AGIUS
INFO: Processing Alex AGIUS SALIBA [Malta -> mt + en]
INFO: Native page not found or empty for Alex AGIUS SALIBA
INFO: Processing Galato ALEXANDRAKI [Greece -> el + en]
INFO: Processing Grégory ALLIONE [France -> fr + en]
INFO: Processing Abir AL-SAHLANI [Sweden -> sv + en]
INFO: Processing Nikolaos ANADIOTIS [Greece -> el + en]
INFO: Native page not found or empty for Nikolaos ANADIOTIS
INFO: English page not found or empty for Nikolaos ANADIOTIS
INFO: Processing Christine ANDERSON [Germany -> de + en]
INFO: Processing Li ANDERSSON [Finland -> fi + en]
INFO: Processing Rasmus ANDRESEN [Germany -> de + en]

Saved 718 rows to mep_wikipedia_pages.csv and mep_wikipedia_pages.xlsx
